# CNU Campus ChatBot — UI 시연 (영상 촬영용)

본 노트북은 **챗봇 UI** 를 띄우기 위한 별도 진입점입니다.
(평가용 진입점은 `src/classifier.ipynb` 와 `chatbot.sh` — PDF p20)

## 사용법
1. 런타임 → T4 GPU 선택
2. Cell 0~2 순서대로 실행
3. iframe 안에서 자유 질의 → 영상 촬영

## 시연 권장 질의 (다양성 보이기)
- 학사 RAG: "졸업 학점 얼마나 들어야 해?"
- URL 안내: "충남대 도서관 위치 알려줘"
- 정중 거절: "충남대 평판 어때?"
- 실시간 학식: "오늘 학식 메뉴 알려줘"


In [ ]:
# [Cell 0] 셋업 — Drive + 의존성 + PROJECT_ROOT 자동 탐지
import os, sys, glob, zipfile, shutil, subprocess
from pathlib import Path

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except ImportError:
    pass

import torch
assert torch.cuda.is_available(), '❌ T4 GPU 선택 필요'
print(f'✅ {torch.cuda.get_device_name(0)} '
      f'({torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB) / '
      f'torch={torch.__version__}')

# PROJECT_ROOT 자동 탐지
_here = Path(os.getcwd()).resolve()
if (_here / 'chatbot.sh').exists():
    PROJECT_ROOT = _here
elif (_here.parent / 'chatbot.sh').exists():
    PROJECT_ROOT = _here.parent
else:
    zips = sorted(glob.glob('/content/drive/MyDrive/Termproject_*final*.zip')
                  or glob.glob('/content/drive/MyDrive/Termproject_*.zip'))
    assert zips, '❌ Drive 에 Termproject_*.zip 없음'
    extract = Path('/content/workspace')
    if extract.exists(): shutil.rmtree(extract)
    with zipfile.ZipFile(zips[-1]) as z: z.extractall(extract)
    PROJECT_ROOT = next((d for d in extract.iterdir() if d.is_dir()), extract)
    print(f'📦 {zips[-1]}')
print(f'📁 PROJECT_ROOT = {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/content/drive/MyDrive/hf_cache/hub'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

subprocess.run([sys.executable,'-m','pip','uninstall','-y','-q',
                'torchao','torchcodec'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

check = subprocess.run(
    [sys.executable, '-c',
     'import fastapi, uvicorn, transformers, faiss, bitsandbytes, '
     'sentence_transformers, peft, accelerate, lxml'],
    capture_output=True)
if check.returncode != 0:
    print('[deps] pip install (~1~2분)')
    subprocess.run(
        [sys.executable,'-m','pip','install','-q','--no-cache-dir',
         '--upgrade-strategy','only-if-needed','--prefer-binary',
         '-r', str(PROJECT_ROOT / 'requirements.txt')])
else:
    print('[deps] 사전설치 충족 — pip 건너뜀')

for m in list(sys.modules):
    if m.startswith(('torchvision','torchaudio','torchao','torchcodec',
                     'transformers','sentence_transformers','timm')):
        del sys.modules[m]

import transformers, bitsandbytes, sentence_transformers, faiss, fastapi
print(f'\n🎉 OK — torch={torch.__version__} / '
      f'transformers={transformers.__version__} / bnb={bitsandbytes.__version__}')


In [ ]:
# [Cell 1] FastAPI 서버 백그라운드 실행 + UI iframe
import subprocess, threading, time, httpx

# 좀비 정리
subprocess.run(['pkill','-9','-f','uvicorn'], capture_output=True)
time.sleep(2)

proc = subprocess.Popen(
    ['python','-m','uvicorn','--app-dir', str(PROJECT_ROOT/'src'),
     'server:app','--host','0.0.0.0','--port','8077'],
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1)

def _pump():
    for line in proc.stdout: print(line, end='')
threading.Thread(target=_pump, daemon=True).start()

print('[wait] 모델 로딩 + 서버 준비 (캐시 hit: 1~2분 / 첫 다운: 5~7분)...')
ready = False
for i in range(900):
    try:
        if httpx.get('http://localhost:8077/health', timeout=2).json().get('ready'):
            ready = True; print(f'\n[ready] {i}s'); break
    except Exception: pass
    time.sleep(1)
assert ready, '❌ 서버가 안 뜸 — 위 로그 확인'

from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(8077, height=720)


In [ ]:
# [Cell 2] (선택) iframe 안에서 자동 batch 시작 안 될 때 수동 트리거
# 영상 촬영에 batch 진행 시연을 포함하고 싶을 때만 실행.
import httpx
r = httpx.post('http://localhost:8077/api/v1/batch/start', timeout=10)
print(r.status_code, r.json())


In [ ]:
# [Cell 3] 영상 촬영 종료 후 서버 종료
try: proc.terminate(); print('[server] terminated')
except Exception as e: print(e)
